# Stage 4 -- Feature Engineering (TF-IDF)

**Goal:** turn `clean_text` (30,000 cleaned documents) into a numeric feature matrix using TF-IDF, ready for dimensionality reduction (Stage 5) and clustering (Stage 6).

### Why TF-IDF specifically?
TF-IDF scores a word highly only when it is frequent *within* a document AND rare *across* the whole collection. This automatically down-weights generic scientific language ("model", "propose", "result") that survived Stage 3's stopword removal, while up-weighting genuinely topic-distinctive terms. This is the standard, well-established baseline for text clustering -- a reasonable and defensible choice to justify in the report, with word embeddings (e.g. Word2Vec, sentence-transformers) mentioned as a possible extension/limitation.

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sp
import joblib

df = pd.read_parquet("../data/processed/arxiv_preprocessed.parquet")
df.shape

(30000, 7)

## Step 1: Configure and fit the vectorizer

**Parameter choices, and why:**
- `min_df=5` -- a term must appear in at least 5 documents to be kept. Filters rare/noisy terms that can't define a *cluster* (a pattern shared across many documents).
- `max_df=0.7` -- a term appearing in more than 70% of documents is dropped. These are effectively domain-wide stopwords (e.g. "model", "method") that add no distinguishing signal.
- `max_features=10000` -- caps the vocabulary at the 10,000 highest-scoring terms, keeping Stage 5 (UMAP/PCA) computationally tractable.
- `ngram_range=(1, 1)` -- unigrams (single words) only, for a first pass. Bigrams (e.g. "neural network" as one token) can capture more specific concepts but multiply the vocabulary size -- worth noting as a possible refinement if time allows.

In [2]:
vectorizer = TfidfVectorizer(
    min_df=5,
    max_df=0.7,
    max_features=10000,
    ngram_range=(1, 1),
)

tfidf_matrix = vectorizer.fit_transform(df["clean_text"])
tfidf_matrix.shape

(30000, 10000)

**Read the shape above:** `(30000, N)` -- 30,000 documents (rows) by N vocabulary terms (columns), where N is at most 10,000 (could be less if fewer terms actually survive the `min_df`/`max_df` filters).

## Step 2: Check sparsity

Confirms *why* we need a sparse matrix format, not a regular dense array -- worth a sentence in your report.

The TF-IDF matrix is 99.09% sparse — only 2,737,032 of 300,000,000 possible entries are non-zero. This is expected: each 150-300 word abstract only uses a tiny fraction of the full 10,000-term vocabulary. Storing this as a dense NumPy array would require roughly 2.4GB of memory (300 million float64 values) despite 99% of that space holding zeros — using scipy.sparse instead stores only the non-zero entries and their positions, making the representation both memory-efficient and fast for the linear algebra operations used in Stage 5

In [3]:
n_nonzero = tfidf_matrix.nnz
n_total = tfidf_matrix.shape[0] * tfidf_matrix.shape[1]
sparsity = 100 * (1 - n_nonzero / n_total)

print(f"Non-zero entries: {n_nonzero:,}")
print(f"Total possible entries: {n_total:,}")
print(f"Sparsity: {sparsity:.2f}% of the matrix is zero")

Non-zero entries: 2,737,032
Total possible entries: 300,000,000
Sparsity: 99.09% of the matrix is zero


## Step 3: Sanity check -- top terms overall

A quick way to verify the vectorizer is behaving sensibly: look at which terms have the highest average TF-IDF score across the whole corpus. These should look like plausible, topic-relevant ML/AI vocabulary -- not noise.

In [4]:
feature_names = np.array(vectorizer.get_feature_names_out())
mean_tfidf_per_term = np.asarray(tfidf_matrix.mean(axis=0)).ravel()

top_n = 30
top_indices = mean_tfidf_per_term.argsort()[::-1][:top_n]

for rank, idx in enumerate(top_indices, start=1):
    print(f"{rank:2d}. {feature_names[idx]:20s} (mean tf-idf: {mean_tfidf_per_term[idx]:.4f})")

 1. data                 (mean tf-idf: 0.0243)
 2. learning             (mean tf-idf: 0.0238)
 3. llm                  (mean tf-idf: 0.0232)
 4. method               (mean tf-idf: 0.0219)
 5. image                (mean tf-idf: 0.0205)
 6. task                 (mean tf-idf: 0.0199)
 7. language             (mean tf-idf: 0.0199)
 8. based                (mean tf-idf: 0.0197)
 9. training             (mean tf-idf: 0.0171)
10. performance          (mean tf-idf: 0.0161)
11. approach             (mean tf-idf: 0.0156)
12. network              (mean tf-idf: 0.0155)
13. framework            (mean tf-idf: 0.0154)
14. agent                (mean tf-idf: 0.0153)
15. system               (mean tf-idf: 0.0150)
16. reasoning            (mean tf-idf: 0.0147)
17. large                (mean tf-idf: 0.0143)
18. feature              (mean tf-idf: 0.0142)
19. multi                (mean tf-idf: 0.0139)
20. time                 (mean tf-idf: 0.0136)
21. across               (mean tf-idf: 0.0131)
22. using    

The top 30 terms by mean TF-IDF score are coherent, topic-relevant ML/AI vocabulary — "learning", "llm", "image", "training", "network", "agent", "reasoning", "benchmark" — with no leftover noise, encoding artifacts, or non-English tokens. This confirms the Stage 3 cleaning pipeline and the min_df=5/max_df=0.7 filtering are working as intended: generic terms weren't so aggressively filtered that meaningful vocabulary was lost, and no obvious junk survived into the final feature set.

## Step 4: Save the matrix and vectorizer for Stage 5

We save two things separately:
- The **matrix** itself (`tfidf_matrix`) -- the actual numeric features, using `scipy.sparse.save_npz` (a format built for sparse matrices; a regular `to_parquet`/`to_csv` would not handle this efficiently or at all).
- The **vectorizer object** (`vectorizer`) -- using `joblib`, so Stage 5/6 (or a future new document) can reuse the exact same vocabulary and IDF weights without refitting from scratch.

In [5]:
sp.save_npz("../data/processed/tfidf_matrix.npz", tfidf_matrix)
joblib.dump(vectorizer, "../data/processed/tfidf_vectorizer.joblib")

# Also save the paper metadata (id, title, categories, etc.) aligned to the same row order,
# so Stage 6/7 can map cluster assignments back to actual papers.
df.to_parquet("../data/processed/arxiv_with_features_meta.parquet")

print("Saved: tfidf_matrix.npz, tfidf_vectorizer.joblib, arxiv_with_features_meta.parquet")

Saved: tfidf_matrix.npz, tfidf_vectorizer.joblib, arxiv_with_features_meta.parquet
